# 🧠 Phase 4: Autonomous Closed-Loop Fuzzy DEMATEL & Causal Triangulation
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection*
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

### 📌 Objectives:
- **Empirical Telemetry Integration**: Automatically binds Phase 2 benchmark results into the 8 DEMATEL causal factors.
- **Fuzzy Inversion & CFCS Defuzzification**: Solve $\tilde{T} = \tilde{X}(I - \tilde{X})^{-1}$ and compute Prominence ($D+R$) and Net Relation ($D-R$).
- **10,000-Iteration Monte Carlo Stability Proof**: Validate ranking invariance under Gaussian boundary noise ($W \ge 0.95$).
- **DirectLiNGAM Causal Triangulation**: Ensure Structural Hamming Distance ($SHD \le 2$).


### 1. ☁️ Google Drive Mount & Project Root Auto-Resolution


In [ ]:
import os, sys
from pathlib import Path

# 1. Mount Google Drive if running in Colab
try:
    from google.colab import drive
    if not Path('/content/drive').exists() and not Path('/content/My Drive').exists():
        drive.mount('/content/drive')
except ImportError:
    print("ℹ️ Running in local/workstation environment.")

# 2. Candidate root paths (prioritizing user's Colab Notebooks directory)
CANDIDATE_ROOTS = [
    Path('/content/My Drive/Colab Notebooks'),
    Path('/content/drive/My Drive/Colab Notebooks'),
    Path('/content/drive/MyDrive/Colab Notebooks'),
    Path('/content/drive/MyDrive/is_ai-vuln'),
    Path('/content/is_ai-vuln'),
    Path('.').resolve()
]

PROJECT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists() and (cand / 'src').exists():
        PROJECT_ROOT = cand.resolve()
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.').resolve()

os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 80)
print(f"✅ Active Project Root : {PROJECT_ROOT}")
print(f"✅ src/ directory found: {(PROJECT_ROOT / 'src').exists()}")
print(f"📁 Data directory      : {(PROJECT_ROOT / 'data').resolve()}")
print("=" * 80)


### 2. 🧮 Autonomous Fuzzy DEMATEL Execution


In [ ]:
import pandas as pd
from src.dematel import (
    run_closed_loop_fuzzy_dematel,
    run_monte_carlo_sensitivity_proof,
    run_causal_triangulation,
    plot_causal_network_digraph
)

dematel_output = run_closed_loop_fuzzy_dematel(beta=0.50)
df_causal = pd.DataFrame(dematel_output["summary_table"])
print(f"Fuzzy DEMATEL Solved! Alpha Threshold: {dematel_output['alpha_threshold']}")
display(df_causal)


### 3. 🎲 10,000-Iteration Monte Carlo Robustness Proof ($W \ge 0.95$)


In [ ]:
L = dematel_output["fuzzy_bounds"]["L"]
M = dematel_output["fuzzy_bounds"]["M"]
U = dematel_output["fuzzy_bounds"]["U"]

mc_res = run_monte_carlo_sensitivity_proof(L, M, U, n_iterations=10000, noise_sigma=0.05)
print(f"🎲 Monte Carlo Sensitivity Proof (10,000 Iterations):")
print(f" - Kendall's W (Prominence): {mc_res['kendalls_w_prominence']} (p = {mc_res['prominence_p_value']:.4e})")
print(f" - Kendall's W (Relation):   {mc_res['kendalls_w_relation']}")
print(f" - Q1 Stability Target Passed: {mc_res['stability_target_met']}")


### 4. 🔗 Algorithmic Causal Triangulation (DirectLiNGAM)

Integrates empirical metric telemetry from Phase 2 benchmark runs if available, else falls back to mock telemetry with clear notification.


In [ ]:
from src.dematel.empirical_mapper import extract_empirical_telemetry_from_experiments

telemetry, is_synthetic_telemetry = extract_empirical_telemetry_from_experiments(
    experiment_dir=PROJECT_ROOT / "experiment_output",
    n_folds=5
)

triangulation_res = run_causal_triangulation(telemetry, dematel_output["adjacency_matrix"])

print(f"🔗 Causal Triangulation Status:")
print(f" - Telemetry Provenance: {'MOCK TELEMETRY (FALLBACK)' if is_synthetic_telemetry else 'REAL EMPIRICAL BENCHMARKS'}")
print(f" - Method: {triangulation_res['triangulation_method']}")
print(f" - Structural Hamming Distance (SHD): {triangulation_res['structural_hamming_distance']} (Target <= {triangulation_res['target_shd_ceiling']})")
print(f" - Triangulation Validated: {triangulation_res['triangulation_passed']}")


### 5. 🕸️ Publication Causal Network Digraph Rendering


In [ ]:
import matplotlib.pyplot as plt

out_dir = PROJECT_ROOT / "experiment_output" / "fuzzy_dematel"
out_dir.mkdir(parents=True, exist_ok=True)

fig_digraph = plot_causal_network_digraph(
    dematel_output, output_filepath=str(out_dir / "figure_causal_network_digraph")
)
plt.show()
